# Pembersihan Data Sayur & Buah

Notebook ini membersihkan file `sayur_buah_kotor.csv` dan menghasilkan `sayur_buah_bersih.csv`.

### Langkah pembersihan:
1. Hapus baris duplikat
2. Tangani nilai tidak valid pada kolom kategorikal (dijadikan NaN)
3. Tangani outlier numerik (dijadikan NaN)
4. Hapus semua baris yang masih mengandung NaN
5. Perbaiki tipe data dan simpan hasil

Setiap langkah disertai laporan singkat untuk memantau perubahan.

In [3]:
import pandas as pd
import numpy as np

INPUT_PATH  = '../dataset/sayur_buah_kotor.csv'
OUTPUT_PATH = '../dataset/sayur_buah_bersih.csv'

# Domain valid untuk pembersihan
VALID_NAMA   = {'Anggur','Wortel','Pisang','Mangga','Jeruk','Apel','Kentang','Cabe','Tomat','Timun'}
VALID_JENIS  = {'Buah','Sayur'}
VALID_LOKASI = {'Pembeku','Pendingin','Suhu Ruang'}
VALID_LABEL  = {'Mentah','Segar','Matang','Busuk','Terlalu Matang'}

HARI_MIN, HARI_MAX = 0, 30
SISA_MIN, SISA_MAX = 0.0, 90.0

df = pd.read_csv(INPUT_PATH)
n_awal = len(df)

print("=" * 60)
print("LAPORAN PEMBERSIHAN DATA")
print("=" * 60)
print(f"\n[AWAL] Jumlah baris   : {n_awal}")
print(f"[AWAL] Jumlah kolom   : {df.shape[1]}")
print(f"\n[AWAL] Missing per kolom:\n{df.isnull().sum()}")

LAPORAN PEMBERSIHAN DATA

[AWAL] Jumlah baris   : 6462
[AWAL] Jumlah kolom   : 6

[AWAL] Missing per kolom:
nama_item               277
jenis_item              276
lokasi_penyimpanan      281
hari_sejak_pembelian    229
sisa_hari               216
label                   399
dtype: int64


## Langkah 1 – Hapus Duplikat
Menghapus baris yang seluruh nilainya sama persis dengan baris lain.

In [4]:
before = len(df)
df = df.drop_duplicates()
n_dup_removed = before - len(df)
print(f"[STEP 1] Duplikat dihapus : {n_dup_removed} baris")

[STEP 1] Duplikat dihapus : 1333 baris


  ## Langkah 2 – Bersihkan Nilai Kategorikal Tidak Valid
  Kolom `nama_item`, `jenis_item`, `lokasi_penyimpanan`, dan `label` hanya boleh berisi nilai tertentu. Nilai di luar daftar yang diizinkan akan diubah menjadi `NaN`.

In [5]:
def clean_categorical(series, valid_set):
    """Set nilai di luar valid_set menjadi NaN."""
    cleaned = series.astype(str).str.strip()
    mask_invalid = ~cleaned.isin(valid_set)
    cleaned[mask_invalid] = np.nan
    return cleaned

df['nama_item']          = clean_categorical(df['nama_item'],          VALID_NAMA)
df['jenis_item']         = clean_categorical(df['jenis_item'],         VALID_JENIS)
df['lokasi_penyimpanan'] = clean_categorical(df['lokasi_penyimpanan'], VALID_LOKASI)
df['label']              = clean_categorical(df['label'],              VALID_LABEL)

n_inval_removed = df[['nama_item','jenis_item','lokasi_penyimpanan','label']].isnull().any(axis=1).sum()
print(f"[STEP 2] Nilai kategorikal tidak valid diubah ke NaN (terdampak {n_inval_removed} baris)")


[STEP 2] Nilai kategorikal tidak valid diubah ke NaN (terdampak 960 baris)


## Langkah 3 – Tangani Outlier Numerik
Nilai `hari_sejak_pembelian` di luar [0,30] dan `sisa_hari` di luar [0.0, 90.0] dianggap tidak wajar, sehingga diubah menjadi `NaN`.

In [6]:
df['hari_sejak_pembelian'] = pd.to_numeric(df['hari_sejak_pembelian'], errors='coerce')
df['sisa_hari']            = pd.to_numeric(df['sisa_hari'],            errors='coerce')

mask_outlier_hari = (df['hari_sejak_pembelian'] < HARI_MIN) | (df['hari_sejak_pembelian'] > HARI_MAX)
mask_outlier_sisa = (df['sisa_hari'] < SISA_MIN)            | (df['sisa_hari'] > SISA_MAX)

n_out_hari = mask_outlier_hari.sum()
n_out_sisa = mask_outlier_sisa.sum()

df.loc[mask_outlier_hari, 'hari_sejak_pembelian'] = np.nan
df.loc[mask_outlier_sisa, 'sisa_hari']            = np.nan

print(f"[STEP 3] Outlier 'hari_sejak_pembelian' (< {HARI_MIN} atau > {HARI_MAX}) : {n_out_hari} baris → NaN")
print(f"[STEP 3] Outlier 'sisa_hari' (< {SISA_MIN} atau > {SISA_MAX})          : {n_out_sisa} baris → NaN")

[STEP 3] Outlier 'hari_sejak_pembelian' (< 0 atau > 30) : 497 baris → NaN
[STEP 3] Outlier 'sisa_hari' (< 0.0 atau > 90.0)          : 570 baris → NaN


## Langkah 4 & 5 – Hapus NaN, Perbaiki Tipe Data, dan Simpan
Semua baris yang masih memiliki nilai kosong (NaN) di kolom mana pun dihapus. Kemudian tipe data numerik disesuaikan: `hari_sejak_pembelian` menjadi integer, `sisa_hari` dibulatkan satu desimal.

In [12]:
before = len(df)
df = df.dropna()
n_null_dropped = before - len(df)
print(f"[STEP 4] Baris dengan nilai null dihapus : {n_null_dropped} baris")

# Perbaiki tipe data
df['hari_sejak_pembelian'] = df['hari_sejak_pembelian'].astype(int)
df['sisa_hari']            = df['sisa_hari'].astype(int) # Mengubah sisa_hari menjadi integer
df = df.reset_index(drop=True)

# Simpan hasil
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nHasil pembersihan disimpan ke: {OUTPUT_PATH}")

[STEP 4] Baris dengan nilai null dihapus : 0 baris

Hasil pembersihan disimpan ke: sayur_buah_bersih.csv


## Ringkasan Akhir
Menampilkan perbandingan jumlah baris awal dan akhir, statistik numerik, dan nilai unik setiap kolom kategorikal untuk memastikan kebersihan data.

In [11]:
print("\n" + "=" * 60)
print("RINGKASAN")
print("=" * 60)
print(f"  Baris awal              : {n_awal}")
print(f"  Duplikat dihapus        : {n_dup_removed}")
print(f"  Null akhir dihapus      : {n_null_dropped}  (termasuk invalid + outlier yang di-NaN)")
print(f"  Baris bersih (output)   : {len(df)}")
print(f"\n  Disimpan ke             : {OUTPUT_PATH}")
print(f"\n[AKHIR] Missing per kolom:\n{df.isnull().sum()}")
print("\n[AKHIR] Statistik numerik:")
display(df[['hari_sejak_pembelian','sisa_hari']].describe().round(2))
print("\n[AKHIR] Nilai unik per kolom kategorikal:")
for col in ['nama_item','jenis_item','lokasi_penyimpanan','label']:
    print(f"  {col}: {sorted(df[col].unique())}")


RINGKASAN
  Baris awal              : 6462
  Duplikat dihapus        : 1333
  Null akhir dihapus      : 1406  (termasuk invalid + outlier yang di-NaN)
  Baris bersih (output)   : 3723

  Disimpan ke             : sayur_buah_bersih.csv

[AKHIR] Missing per kolom:
nama_item               0
jenis_item              0
lokasi_penyimpanan      0
hari_sejak_pembelian    0
sisa_hari               0
label                   0
dtype: int64

[AKHIR] Statistik numerik:


,hari_sejak_pembelian,sisa_hari
count,3723.00,3723.00
mean,2.00,11.49
std,1.79,10.71
min,0.00,0.00
25%,1.00,2.60
50%,2.00,8.20
75%,3.00,17.30
max,6.00,42.00



[AKHIR] Nilai unik per kolom kategorikal:
  nama_item: ['Anggur', 'Apel', 'Cabe', 'Jeruk', 'Kentang', 'Mangga', 'Pisang', 'Timun', 'Tomat', 'Wortel']
  jenis_item: ['Buah', 'Sayur']
  lokasi_penyimpanan: ['Pembeku', 'Pendingin', 'Suhu Ruang']
  label: ['Busuk', 'Matang', 'Mentah', 'Segar', 'Terlalu Matang']
